# Phase 3 — Reranker conversion diagnosis (Option-0, slot-free)

Decomposes the dev **final-turn** nDCG@20 gap into **recall-bound** (gold never reached the top-500 pool → ColBERT/recall lever) vs **ranking-bound** (gold in the pool but not ranked top-20 → K2/cross-encoder lever). Shares the channel + ColBERT setup with `phase3_blindA_submission`; trains nothing. Run top-to-bottom.

Loads pretrained artifacts: K2 (`K2_MODEL_PATH` from `phase2_rerank.ipynb`), the ColBERT checkpoint (`COLBERT_OUT_DIR` from `phase2_colbert_finetune.ipynb`).

## 1. Drive + HF auth (Colab Secrets)

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import os
DRIVE='/content/drive/MyDrive/recsys2026'
os.environ['HF_HOME']=f'{DRIVE}/hf_cache'; OUT=f'{DRIVE}/outputs'
os.makedirs(os.environ['HF_HOME'],exist_ok=True); os.makedirs(OUT,exist_ok=True)
try:
    t=userdata.get('HF_TOKEN'); os.environ['HF_TOKEN']=os.environ['HUGGINGFACE_HUB_TOKEN']=t
    from huggingface_hub import login; login(t); print('HF ok')
except Exception as e: print('no HF_TOKEN secret:', e)
try:                                   # only needed when RESPONDER=='gemini'
    os.environ['GEMINI_API_KEY']=os.environ.get('GEMINI_API_KEY') or userdata.get('GEMINI_API_KEY'); print('Gemini key ok')
except Exception as e: print('no GEMINI_API_KEY secret (only needed for the responder):', e)

## 2. Clone + install

In [ ]:
!git clone --branch fresh-start --depth 1 https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026 2>/dev/null || (cd /content/recsys2026 && git pull)
%cd /content/recsys2026
!pip -q install datasets bm25s scipy scikit-learn lightgbm sentence-transformers numpy pandas pylate peft google-genai
!pip -q uninstall -y torchao   # transformers wants torchao>0.16 but Colab ships 0.10 and its check RAISES; we don't use it
import sys; sys.path.insert(0,'.')

In [ ]:
# OOM hygiene — MUST run before any import that pulls in JAX/torch (RESTART to apply mid-session).
# Root cause of the GPU OOMs: JAX preallocates 75% of VRAM on init, leaving PyTorch ~24% -> OOM.
import os
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'   # JAX: allocate on demand (THE fix)
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'
os.environ['TORCHDYNAMO_DISABLE'] = '1'
print('OOM-hygiene env set — RESTART if torch/jax were already imported')

## 3. Config (shared with the submission nb; only channel knobs matter here)

In [ ]:
# ── mode ──
BLIND=False                # this is the slot-free DEV experiments nb — keep False (no submission path here)
RESPONDER='stub'           # 'stub' (default predicted_response, read nDCG axis only) | 'gemini' (full composite)
DEFAULT_RESPONSE='ok'      # stub responder value written to every row (nDCG-only online eval; nb82 convention)
SMOKE=0                    # >0 caps serve turns for a quick pipeline check (0 = all)
SEED=42

# ── splits ──
ORG='talkpl-ai'
DEV_SESSIONS=1000          # dev sessions probed by the experiment cells (serve set / recall sweep)
BLIND_DATASET=f'{ORG}/TalkPlayData-Challenge-Blind-A'   # 80 sessions, NO gold

# ── retrieval ──
TOPK=500                   # fused-pool depth into the reranker
SUBMIT_K=20                # ids per submission row (official cap)
DENSE_MODEL='BAAI/bge-large-en-v1.5'
DENSE_QUERY_PREFIX='Represent this sentence for searching relevant passages: '
CONTENT_MODALITIES={'cknn_audio':'audio-laion_clap','cknn_attr':'attributes-qwen3_embedding_0.6b'}
ENRICHED_GLOB=f'{OUT}/catalog_enriched_*.parquet'

# ── ColBERT (the LoRA fine-tune from phase2_colbert_finetune) ──
# D_LEN / EXPANSION_FIRST MUST match phase2_colbert_finetune (the checkpoint was trained + indexed at these).
COLBERT_OUT_DIR=f'{OUT}/colbert/music-colbert-v1'   # merged plain-ColBERT checkpoint
Q_LEN=96; D_LEN=512; BSIZE=128; EXPANSION_FIRST=True
FT_IDX_FOLDER=f'{OUT}/colbert_plaid_ft'; FT_IDX_BASE='music-colbert-v1-d%d'%D_LEN

# ── K2 (LGBM) — LOAD ONLY. This notebook trains NOTHING; train K2 in phase2_rerank.ipynb. ──
# Point at the trained artifact (.txt + .features.json sidecar) if an experiment needs reranked scores.
# If that K2 used the frozen-CE feature, score the serve-side CE over the serve pool only (no train).
K2_MODEL_PATH=f'{OUT}/k2_lgbm.txt'
CE_MODEL='BAAI/bge-reranker-v2-m3'; CROSS_ENCODER_K=50; CE_MAX_DOC_TOKENS=480   # used only if K2 needs ce_score

import random as _r, numpy as _np
_r.seed(SEED); _np.random.seed(SEED)
try:
    import torch as _t; _t.manual_seed(SEED); _t.cuda.manual_seed_all(SEED)
except Exception: pass
print('MODE: DEV experiments (slot-free, trains nothing) | responder', RESPONDER, '| D_LEN', D_LEN, '| smoke', SMOKE)

## 4. Catalog + base channels + dense doc_mat

In [ ]:
import glob, os, pickle, hashlib, pandas as pd, numpy as np
from datasets import load_dataset
from mcrs.data.catalog import Catalog
from mcrs.data.embeddings import TrackEmbeddings, UserEmbeddings
from mcrs.data.conversations import Conversations
from sentence_transformers import SentenceTransformer
from mcrs.retrieval.query import QueryBuilder
from mcrs.retrieval.bm25_channel import BM25Channel
from mcrs.retrieval.dense_channel import DenseChannel
from mcrs.retrieval.personalization import ContentKNNChannel, CFChannel, SameArtistChannel
from mcrs.retrieval.related_artist import RelatedArtistChannel, build_artist_cooc, tid_to_artists_from_catalog
from mcrs.retrieval.fusion import RRFFusion

meta_rows=load_dataset(f'{ORG}/TalkPlayData-Challenge-Track-Metadata',split='all_tracks')
_enr=sorted(glob.glob(ENRICHED_GLOB)); assert _enr, f'No enriched parquet at {ENRICHED_GLOB} — run A1 first'
_edf=pd.read_parquet(_enr[-1]); enr=dict(zip(_edf['track_id'],_edf['enriched_doc']))
cat=Catalog(meta_rows,enriched_docs=enr); USE_ENRICHED=True
print(f'enriched docs {len(enr)} (from {_enr[-1].split("/")[-1]})')

tre=load_dataset(f'{ORG}/TalkPlayData-Challenge-Track-Embeddings',split='all_tracks')
_avail=set(tre.column_names); CKNN_MODS={l:m for l,m in CONTENT_MODALITIES.items() if m in _avail}
te={l:TrackEmbeddings(tre.select_columns(['track_id',m]),modalities=[m]) for l,m in CKNN_MODS.items()}
te_cf=TrackEmbeddings(tre.select_columns(['track_id','cf-bpr']),modalities=['cf-bpr'])
ued=load_dataset(f'{ORG}/TalkPlayData-Challenge-User-Embeddings'); ue=UserEmbeddings([r for sp in ued for r in ued[sp]])
dsd=load_dataset(f'{ORG}/TalkPlayData-Challenge-Dataset')

model=SentenceTransformer(DENSE_MODEL,device='cuda')
# config-hashed cache key (model + enriched version + catalog size) — never silently load a stale matrix
_dm_sig=hashlib.md5(f'{DENSE_MODEL}|enriched={USE_ENRICHED}|{os.path.basename(_enr[-1])}|n={len(cat.index_to_id)}'.encode()).hexdigest()[:8]
DOC_MAT_NPY=f'{OUT}/dense_doc_mat_{_dm_sig}.npy'
if os.path.exists(DOC_MAT_NPY):
    doc_mat=np.load(DOC_MAT_NPY); print('loaded cached doc_mat', doc_mat.shape)
else:
    doc_mat=model.encode([cat.id_to_metadata(t,enriched=USE_ENRICHED) for t in cat.index_to_id],batch_size=256,normalize_embeddings=True,show_progress_bar=True)
    np.save(DOC_MAT_NPY, doc_mat); print('cached doc_mat ->', DOC_MAT_NPY)
dense=DenseChannel(cat.index_to_id,doc_mat,lambda qs:model.encode([DENSE_QUERY_PREFIX+q for q in qs],batch_size=256,normalize_embeddings=True),normalize=False)
COOC_PKL=f'{OUT}/artist_cooc.pkl'
cooc=pickle.load(open(COOC_PKL,'rb')) if os.path.exists(COOC_PKL) else build_artist_cooc(dsd['train'],tid_to_artists_from_catalog(cat))
if not os.path.exists(COOC_PKL): pickle.dump(cooc,open(COOC_PKL,'wb'))
cknn=[ContentKNNChannel(te[l],m,label=l) for l,m in CKNN_MODS.items()]
base_chans=[BM25Channel(cat,enriched=USE_ENRICHED), dense, *cknn,
            CFChannel(ue,te_cf,'cf-bpr'), SameArtistChannel(cat), RelatedArtistChannel(cat,cooc)]
qb_full=QueryBuilder()                    # base channels' serve query (full history)
qb_focused=QueryBuilder(recency_window=1) # ColBERT's focused query (train==serve)
print('base channels:', [c.label for c in base_chans])

## 5. ColBERT PLAID channel (fine-tuned, focused-query routed)

In [ ]:
from pylate import models
from mcrs.training.colbert_index import build_or_load_plaid, colbert_retrieve
from mcrs.retrieval.colbert_channel import colbert_doc_text
import hashlib

assert os.path.isdir(COLBERT_OUT_DIR), f'ColBERT checkpoint missing at {COLBERT_OUT_DIR} — run phase2_colbert_finetune first'
ft=models.ColBERT(model_name_or_path=COLBERT_OUT_DIR, query_length=Q_LEN, document_length=D_LEN)
doc_fn=lambda t: colbert_doc_text(cat, t, expansion_first=EXPANSION_FIRST)
# checkpoint-keyed PLAID index name so a retrain can't reuse a stale index (mismatched embedding spaces)
_ck_sig=hashlib.md5('|'.join(f'{f}:{os.stat(os.path.join(rt,f)).st_size}:{int(os.stat(os.path.join(rt,f)).st_mtime)}'
                             for rt,_,fs in os.walk(COLBERT_OUT_DIR) for f in sorted(fs)).encode()).hexdigest()[:8]
# doc-side sig: enriched catalog + D_LEN + EXPANSION_FIRST determine the doc vectors (same format as the
# fine-tune gate + blindA so the index is shared; a new parquet / D_LEN / expansion order busts it).
_doc_sig=hashlib.md5(f'{os.path.basename(_enr[-1])}|n={len(cat.index_to_id)}|d{D_LEN}|ef{EXPANSION_FIRST}'.encode()).hexdigest()[:8]
retr=build_or_load_plaid(ft, cat, FT_IDX_FOLDER, f'{FT_IDX_BASE}-{_ck_sig}-{_doc_sig}', doc_fn, batch_size=BSIZE)

class PlaidColBERTChannel:
    """Fusion channel backed by the fine-tuned ColBERT PLAID index. query_key='colbert' so the harness /
    build_rerank_groups route it the focused query while base channels keep the full query."""
    label='colbert'; query_key='colbert'
    def __init__(self, model, retriever): self.model, self.retriever = model, retriever
    def batch_text_to_item_retrieval(self, queries, topk, batch_context=None, user_ids=None):
        if not queries: return []
        return colbert_retrieve(self.model, self.retriever, list(queries), topk, batch_size=BSIZE)

colbert=PlaidColBERTChannel(ft, retr)
chans=base_chans+[colbert]
fusion=RRFFusion(chans, k=60)
labels=[c.label for c in chans]
PCQ={'colbert': qb_focused}               # per-channel routing: ColBERT gets the focused query
print('fusion channels:', labels)

## Diagnosis
Reconstructs the production K2 serve spine (mirrors `phase3_blindA`) and reranks the fused pool per final-turn target, then prints fusion-only vs +K2 conversion and the recall-/ranking-bound verdict.

In [ ]:
# === EXPERIMENT 2 — RERANKER CONVERSION DIAGNOSIS (Option-0, slot-free) ===
# Of the golds that ARE in the fused top-500 pool, what fraction does K2 land in the top-20? Splits the
# nDCG@20 miss budget into recall_loss (pool miss -> ColBERT) vs ranking_loss (mis-ranked -> K2/CE).
# Run the setup cells (catalog + channels + ColBERT) above first.
import json
from mcrs.data.conversations import Conversations
from mcrs.rerank.features import FeatureBuilder
from mcrs.rerank.lgbm import LGBMReranker
from mcrs.eval.diagnostics import conversion_diagnosis

conv_dev = Conversations(dsd['test'].select(range(min(DEV_SESSIONS, len(dsd['test'])))))
ft = list(conv_dev.gold_target_turns())            # final-turn proxy (trailing gold-bearing turn/session)
golds = [conv_dev.gold(t.session_id, t.turn_number) for t in ft]
segs  = [t.segment for t in ft]
print(f'final-turn proxy: {len(ft)} turns | warm {segs.count("warm")} / cold {segs.count("cold")}')

# routed serve pools (ColBERT focused, base channels full) — exactly what K2 reranks at serve
serve_bc = [{'history_tids': t.history_tids, 'user_id': t.user_id} for t in ft]
serve_pools = fusion.fuse([qb_full.build(t).text for t in ft], TOPK, batch_context=serve_bc,
                          user_ids=[t.user_id for t in ft],
                          per_channel_queries={'colbert': [qb_focused.build(t).text for t in ft]})

# K2 feature reconstruction (mirror phase3_blindA): dense_cos + (if trained with it) the frozen-CE score
feat_names = json.load(open(K2_MODEL_PATH + '.features.json'))
need_ce = ('ce_score' in feat_names)
print('K2 features:', len(feat_names), '| needs ce_score:', need_ce)

_qcache = {}
_qvm = model.encode([DENSE_QUERY_PREFIX + qb_full.build(t).text for t in ft], batch_size=256, normalize_embeddings=True)
_qcache.update({(t.session_id, t.turn_number): _qvm[i] for i, t in enumerate(ft)})
def dense_cos(ctx, tid):
    j = cat.id_to_index.get(tid)
    if j is None: return 0.0
    v = _qcache.get((ctx.session_id, ctx.turn_number))
    return float(v @ doc_mat[j]) if v is not None else 0.0
score_fns = {'dense_cos': dense_cos}

if need_ce:
    from mcrs.rerank.neural import NeuralReranker, build_ce_score_lookup, make_ce_feature_fn
    from mcrs.rerank.cross_encoder import build_cross_encoder_score_fn
    _fce = build_cross_encoder_score_fn(CE_MODEL, device='cuda', max_length=512, max_doc_tokens=CE_MAX_DOC_TOKENS, dtype='fp16')
    _nr = NeuralReranker(cat, qb_full, _fce, cross_encoder_k=CROSS_ENCODER_K, enriched=USE_ENRICHED)
    print('scoring frozen CE over the serve pools...')
    ce_lookup = build_ce_score_lookup(ft, serve_pools, _nr, normalize=True, show_progress=True)
    score_fns['ce_score'] = make_ce_feature_fn(ce_lookup, default=-1.0)

fb = FeatureBuilder(cat, labels, score_fns=score_fns)
k2 = LGBMReranker(fb).load(K2_MODEL_PATH)
k2.assert_feature_parity()                          # early train==serve guard (names any missing feature)
print('K2 loaded + parity OK |', len(fb.feature_names), 'features')

# fusion-only (RRF order) vs K2-reranked full lists
fused_ids    = [[c.track_id for c in pool] for pool in serve_pools]
reranked_ids = [[c.track_id for c in k2.rerank(t, list(pool)).items] for t, pool in zip(ft, serve_pools)]

fus = conversion_diagnosis(fused_ids, golds, pool_k=TOPK, top_k=20, segments=segs)
k2d = conversion_diagnosis(reranked_ids, golds, pool_k=TOPK, top_k=20, segments=segs)

def _show(tag, d):
    print(f'\n[{tag}]  recall@{TOPK}={d["recall_at_pool"]:.3f}  recall@20={d["recall_at_top"]:.3f}  '
          f'conversion={d["conversion"]:.3f}')
    print(f'         recall_loss={d["recall_loss"]:.3f} (pool miss)  '
          f'ranking_loss={d["ranking_loss"]:.3f} (in-pool, mis-ranked)  -> {d["verdict"].upper()}')
    for s, b in d.get('by_segment', {}).items():
        print(f'         {s}: recall@{TOPK}={b["recall_at_pool"]:.3f} recall@20={b["recall_at_top"]:.3f} '
              f'conversion={b["conversion"]:.3f}')
_show('fusion-only', fus)
_show('+K2',         k2d)

print('\nREAD: RANKING-bound -> spend on K2 / cross-encoder (in-pool golds not reaching top-20).')
print('      RECALL-bound  -> spend on ColBERT / recall (golds never reach the pool; bounded by the wall).')
print(f'      K2 conversion {fus["conversion"]:.3f} -> {k2d["conversion"]:.3f}  '
      f'(recall@20 {fus["recall_at_top"]:.3f} -> {k2d["recall_at_top"]:.3f}).')
